# PlantSeg Multi-Seed Retraining -- Classifiers-Only Lane B (seed 7)

One of **2 parallel Kaggle notebooks** covering the classifier half of the 2-new-seed
(3 total, including the original SEED=42 checkpoints) retraining pass for the Array revision's
variance/robustness response to R4/R5. This is a **replacement** for the classifier sections of
the original 6-lane split, which failed on every lane due to a `DATA_ROOT` bug in the classifier
config cell (see below) -- **the segmenter half of that original run already completed
successfully and is not repeated here.**

**This notebook (Lane B, seed 7) trains all 8 classifiers at seed 7:**
ConvNeXtV2Tiny, EfficientNetV2S, ResNet50, EfficientNetB0, MobileNetV3Large, ViTBase16, SwinTiny, GatedFusionLight.
Hyperparameters, architectures, losses and schedulers are all unchanged from the original Stage 1
trainer -- only `SEED` and the output directory differ, so these results are directly comparable
to the existing SEED=42 checkpoints.

**Estimated time:** ~7.7h of core training (sum of each classifier's original single-seed
`train_time_sec` from its `result_*.pkl`), plus data-loading/TTA-eval/deployment-benchmark
overhead -- budget **~9-9.5h**, leaving ~2.5-3h of margin under Kaggle's 12h session limit, in
line with the original 6-lane split's margin philosophy. All 8 classifiers fit in one seed's
lane; the only reason this is split into 2 notebooks rather than 1 is that seed 7 and seed
123 together (~15.5h raw) would not fit in a single 12h session.

**What was fixed vs. the original classifier-section bug:** the original notebooks hardcoded
`DATA_ROOT = Path("/kaggle/input/plantseg/plantseg")` in the classifier config cell, which does
not exist on Kaggle for this dataset attachment -- the real mount path (confirmed from the
segmenter section of the same lanes, which someone had already manually corrected and which DID
run to completion) is `/kaggle/input/datasets/alenmaruf/plantseg/plantseg`. This notebook uses
that verified path as primary, with the old paths kept as fallbacks.

**Attach as a Kaggle dataset:** the PlantSeg image+annotation dataset (Metadata.csv + images/ +
annotation_train.json/annotation_val.json/annotation_test.json). No pretrained checkpoints need
attaching -- classifiers train from ImageNet initialization via `torchvision`/`timm`, same as the
original runs.

**After it finishes:** collect this notebook's output folder (`/kaggle/working/s1_results_seed7/`)
and bring it back to `revision_work/multiseed/` alongside the already-collected seed-7
segmenter results, so both halves of seed 7 can be aggregated together.

In [1]:
# ============ CLASSIFIER SECTION: IMPORTS + SEED ============
!pip install -q timm
import os, json, math, time, random, warnings, pickle
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision; from torchvision import transforms
import timm
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
                             recall_score, matthews_corrcoef, confusion_matrix, roc_auc_score)
warnings.filterwarnings("ignore")

SEED = 7   # <-- new seed for this run (original paper checkpoints used SEED=42)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:", device, "| SEED:", SEED)
assert device.type == "cuda", "No GPU detected -- set Settings > Accelerator (T4 x2 / P100) before Run All."

Device: cuda | SEED: 7


In [2]:
# ============ CLASSIFIER SECTION: CONFIG ============
# DATA_ROOT fixed: the original classifier-section block hardcoded a path that does not exist on
# Kaggle for this dataset attachment ("/kaggle/input/plantseg/plantseg"); the verified correct
# path (confirmed by the segmenter section of the same lanes completing successfully) is used as
# primary here, with the old paths kept as fallbacks.
DATA_ROOT = Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"), Path("/kaggle/input/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): DATA_ROOT=cand; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not found under {DATA_ROOT} -- edit DATA_ROOT."

OUTDIR = Path(f"/kaggle/working/s1_results_seed{SEED}"); FIG = OUTDIR/"figures"
OUTDIR.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

MODELS_TO_TRAIN = ['ConvNeXtV2Tiny', 'EfficientNetV2S', 'ResNet50', 'EfficientNetB0', 'MobileNetV3Large', 'ViTBase16', 'SwinTiny', 'GatedFusionLight']   # all 8 classifiers -- this lane's full assignment

SCOPE = "full"
FRUIT_HOSTS = {"Apple","Citrus","Grape","Banana","Peach","Blueberry",
               "Plum","Cherry","Raspberry","Strawberry"}

NUM_EPOCHS      = 40
PATIENCE        = 8
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
USE_PRETRAINED  = True
USE_CLASS_WEIGHTS = True
LABEL_SMOOTHING = 0.05
IMG_SIZE_CNN    = 384   # lower to 320/256 if OOM -- CNN classifiers trained at 384x384
USE_RANDAUG     = True
USE_TTA         = True
QUICK_SMOKE     = False

print("DATA_ROOT:", DATA_ROOT); print(f"Models to train (seed {SEED}):", MODELS_TO_TRAIN)

DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
Models to train (seed 7): ['ConvNeXtV2Tiny', 'EfficientNetV2S', 'ResNet50', 'EfficientNetB0', 'MobileNetV3Large', 'ViTBase16', 'SwinTiny', 'GatedFusionLight']


In [3]:
# ============ CLASSIFIER SECTION: DATA INDEX ============
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["label"] = meta["Disease"]; meta["plant"] = meta["Plant"]
if SCOPE == "fruit":
    meta = meta[meta["plant"].isin(FRUIT_HOSTS)].reset_index(drop=True)
meta = meta[meta["filepath"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)

counts = train_df["label"].map(label2idx).value_counts().reindex(range(NUM_CLASSES)).fillna(0)
inv = np.where(counts.values > 0, 1.0/counts.values, 0.0)
cw  = inv / inv[inv>0].mean()
class_weights = torch.tensor(cw, dtype=torch.float32, device=device)

print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

classes=115  train=5367  val=846  test=1561


In [4]:
# ============ CLASSIFIER SECTION: TRANSFORMS / DATASET ============
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

def make_transforms(sz):
    ops = [transforms.RandomResizedCrop(sz, scale=(0.7,1.0)),
           transforms.RandomHorizontalFlip(),
           transforms.RandomVerticalFlip(p=0.2),
           transforms.RandAugment(num_ops=2, magnitude=9),
           transforms.ToTensor(), transforms.Normalize(MEAN, STD),
           transforms.RandomErasing(p=0.25)]
    ev = transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return transforms.Compose(ops), ev

class DS(Dataset):
    def __init__(self, d, t):
        self.df = d.reset_index(drop=True); self.t = t
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.t(Image.open(r.filepath).convert("RGB")), label2idx[r.label]

In [5]:
# ============ CLASSIFIER SECTION: MODEL ZOO ============
# Verbatim from Stage 1's build_backbone() -- NOTE GatedFusionLight is intentionally NOT a case
# here: it is handled via FUSION_MODELS below and hardcoded to img_size=224 in train_one(),
# matching how it was actually trained.
def build_backbone(name, num_classes, pretrained=USE_PRETRAINED):
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2" if pretrained else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m, IMG_SIZE_CNN, 24
    if name == "EfficientNetB0":
        return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 24
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m, IMG_SIZE_CNN, 16
    if name == "EfficientNetV2S":
        return timm.create_model("tf_efficientnetv2_s", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 12
    if name == "ConvNeXtV2Tiny":
        return timm.create_model("convnextv2_tiny", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 16
    if name == "ViTBase16":
        return timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=num_classes), 224, 16
    if name == "SwinTiny":
        return timm.create_model("swin_tiny_patch4_window7_224", pretrained=pretrained, num_classes=num_classes), 224, 16
    raise ValueError(name)

class _Gated(nn.Module):
    def __init__(self, ba, da, bb, db, nc, gh, hh, dr):
        super().__init__(); self.branch_a=ba; self.branch_b=bb; fused=da+db
        self.gate = nn.Sequential(nn.Linear(fused,gh), nn.ReLU(), nn.Linear(gh,2), nn.Softmax(dim=1))
        self.classifier = nn.Sequential(nn.Linear(fused,hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh,nc))
    def forward(self, x):
        fa, fb = self.branch_a(x), self.branch_b(x)
        g = self.gate(torch.cat([fa,fb], 1))
        return self.classifier(torch.cat([fa*g[:,0:1], fb*g[:,1:2]], 1))

def GatedFusionLightModel(nc, pretrained=USE_PRETRAINED):
    mb = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2" if pretrained else None)
    da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
    b  = timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=0, global_pool="avg")
    return _Gated(mb, da, b, b.num_features, nc, 128, 256, 0.3)

FUSION_MODELS = {"GatedFusionLight": GatedFusionLightModel}

In [6]:
# ============ CLASSIFIER SECTION: TRAIN / METRICS HELPERS ============
def train_one(model_name, num_epochs=NUM_EPOCHS, patience=PATIENCE):
    if model_name in FUSION_MODELS:
        model = FUSION_MODELS[model_name](NUM_CLASSES).to(device); img_size, bs = 224, 16
    else:
        model, img_size, bs = build_backbone(model_name, NUM_CLASSES); model = model.to(device)
    ttf, etf = make_transforms(img_size)
    tl = DataLoader(DS(train_df, ttf), batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(DS(val_df,   etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(DS(test_df,  etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    is_tf = model_name in ("ViTBase16","SwinTiny")
    lr = 5e-5 if is_tf else LR; warm_ep = 3 if is_tf else 2; pat = 10 if is_tf else patience
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    warm = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=warm_ep)
    cos  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, num_epochs-warm_ep))
    sch  = torch.optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[warm_ep])
    crit = nn.CrossEntropyLoss(weight=class_weights if USE_CLASS_WEIGHTS else None, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}; best=-1; best_state=None; bad=0
    for ep in range(num_epochs):
        model.train(); run=cor=tot=0
        for x,y in tl:
            x,y = x.to(device),y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()*x.size(0); cor += (out.argmax(1)==y).sum().item(); tot += y.size(0)
        trl, tra = run/tot, cor/tot
        model.eval(); vr=vc=vt=0
        with torch.no_grad():
            for x,y in vl:
                x,y = x.to(device),y.to(device); out = model(x); loss = crit(out,y)
                vr += loss.item()*x.size(0); vc += (out.argmax(1)==y).sum().item(); vt += y.size(0)
        vlo, vac = vr/vt, vc/vt; sch.step()
        for k,v in zip(hist, [trl,vlo,tra,vac]): hist[k].append(v)
        print(f"[{model_name}] ep{ep+1}/{num_epochs} lr={opt.param_groups[0]['lr']:.1e} "
              f"trL={trl:.3f} vL={vlo:.3f} trA={tra:.3f} vA={vac:.3f}")
        if vac > best+1e-4:
            best=vac; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if ep >= warm_ep and bad >= pat:
                print(f"  early stop @ ep{ep+1}"); break
    model.load_state_dict(best_state); torch.save(best_state, OUTDIR/f"{model_name}_seed{SEED}_best.pt")
    model.eval(); P=[]; Y=[]; PR=[]
    with torch.no_grad():
        for x,y in te:
            x = x.to(device); lg = model(x)
            if USE_TTA: lg = lg + model(torch.flip(x, dims=[3]))
            p = F.softmax(lg, 1).cpu().numpy(); P.extend(p.argmax(1)); Y.extend(y.numpy()); PR.extend(p)
    return model, img_size, hist, np.array(Y), np.array(P), np.array(PR)

def deployment(model, img_size, nw=5, ni=30):
    n = sum(p.numel() for p in model.parameters())
    tmp = OUTDIR/"_t.pt"; torch.save(model.state_dict(), tmp)
    sz = os.path.getsize(tmp)/1e6; os.remove(tmp)
    d  = torch.randn(1, 3, img_size, img_size)
    def t(dev):
        m = model.to(dev).eval(); dd = d.to(dev)
        with torch.no_grad():
            for _ in range(nw): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(ni): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            return (time.time()-t0)/ni*1000
    lg = t(device) if device.type=="cuda" else float("nan")
    lc = t(torch.device("cpu")); model.to(device)
    return {"params_M": n/1e6, "checkpoint_size_MB": sz, "latency_cpu_ms": lc, "latency_gpu_ms": lg}

def compute_metrics(y, yp, ypb, n):
    cm = confusion_matrix(y, yp, labels=range(n)); specs=[]
    for i in range(n):
        tn = cm.sum()-cm[i,:].sum()-cm[:,i].sum()+cm[i,i]; fp = cm[:,i].sum()-cm[i,i]
        specs.append(tn/(tn+fp) if (tn+fp)>0 else float("nan"))
    present = [c for c in range(n) if (y==c).sum()>0]; aucs=[]
    for c in present:
        try: aucs.append(roc_auc_score((y==c).astype(int), ypb[:,c]))
        except Exception: pass
    return {"accuracy": accuracy_score(y,yp),
            "balanced_accuracy": balanced_accuracy_score(y,yp),
            "f1_macro": f1_score(y,yp,average="macro"),
            "f1_weighted": f1_score(y,yp,average="weighted"),
            "precision_macro": precision_score(y,yp,average="macro",zero_division=0),
            "recall_macro": recall_score(y,yp,average="macro",zero_division=0),
            "mcc": matthews_corrcoef(y,yp),
            "roc_auc_macro": float(np.mean(aucs)) if aucs else float("nan")}, cm

In [7]:
# ============ CLASSIFIER SECTION: MAIN LOOP (all 8 classifiers, SEED 7) ============
for MODEL_NAME in MODELS_TO_TRAIN:
    print(f"\n{'='*60}\n  Training: {MODEL_NAME}  (seed {SEED})\n{'='*60}\n")
    t0 = time.time()
    model, img_size, hist, Y, P, PR = train_one(MODEL_NAME)
    train_time = time.time()-t0
    metrics, cm = compute_metrics(Y, P, PR, NUM_CLASSES)
    dep = deployment(model, img_size)
    result = {"model_name": MODEL_NAME, "seed": SEED, "history": hist,
              "y_true": Y, "y_pred": P, "y_prob": PR,
              "metrics": metrics, "cm": cm, "deployment": dep,
              "img_size": img_size, "train_time_sec": train_time, "class_names": CLASS_NAMES}
    pickle.dump(result, open(OUTDIR/f"result_{MODEL_NAME}_seed{SEED}.pkl", "wb"))
    print("SAVED", OUTDIR/f"result_{MODEL_NAME}_seed{SEED}.pkl")
    print({k: round(v,4) for k,v in metrics.items() if isinstance(v, float)})
    print(f"  train_time: {train_time/3600:.2f}h")
print("\nAll 8 classifiers done for this lane (seed 7).")


  Training: ConvNeXtV2Tiny  (seed 7)



model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

[ConvNeXtV2Tiny] ep1/40 lr=1.6e-04 trL=4.170 vL=3.212 trA=0.253 vA=0.505
[ConvNeXtV2Tiny] ep2/40 lr=3.0e-04 trL=2.580 vL=2.345 trA=0.511 vA=0.604
[ConvNeXtV2Tiny] ep3/40 lr=3.0e-04 trL=2.297 vL=2.707 trA=0.541 vA=0.515
[ConvNeXtV2Tiny] ep4/40 lr=3.0e-04 trL=1.983 vL=2.239 trA=0.628 vA=0.617
[ConvNeXtV2Tiny] ep5/40 lr=3.0e-04 trL=1.674 vL=2.206 trA=0.703 vA=0.649
[ConvNeXtV2Tiny] ep6/40 lr=2.9e-04 trL=1.441 vL=2.109 trA=0.786 vA=0.673
[ConvNeXtV2Tiny] ep7/40 lr=2.9e-04 trL=1.331 vL=2.126 trA=0.812 vA=0.697
[ConvNeXtV2Tiny] ep8/40 lr=2.8e-04 trL=1.279 vL=2.157 trA=0.835 vA=0.686
[ConvNeXtV2Tiny] ep9/40 lr=2.8e-04 trL=1.277 vL=2.326 trA=0.840 vA=0.635
[ConvNeXtV2Tiny] ep10/40 lr=2.7e-04 trL=1.274 vL=2.213 trA=0.845 vA=0.656
[ConvNeXtV2Tiny] ep11/40 lr=2.6e-04 trL=1.219 vL=2.314 trA=0.856 vA=0.644
[ConvNeXtV2Tiny] ep12/40 lr=2.5e-04 trL=1.152 vL=2.066 trA=0.882 vA=0.701
[ConvNeXtV2Tiny] ep13/40 lr=2.4e-04 trL=1.062 vL=2.177 trA=0.916 vA=0.686
[ConvNeXtV2Tiny] ep14/40 lr=2.3e-04 trL=1.054 v

model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

[EfficientNetV2S] ep1/40 lr=1.6e-04 trL=5.099 vL=4.565 trA=0.045 vA=0.135
[EfficientNetV2S] ep2/40 lr=3.0e-04 trL=3.261 vL=2.629 trA=0.361 vA=0.572
[EfficientNetV2S] ep3/40 lr=3.0e-04 trL=2.404 vL=2.466 trA=0.540 vA=0.602
[EfficientNetV2S] ep4/40 lr=3.0e-04 trL=1.906 vL=2.345 trA=0.669 vA=0.632
[EfficientNetV2S] ep5/40 lr=3.0e-04 trL=1.669 vL=2.309 trA=0.743 vA=0.675
[EfficientNetV2S] ep6/40 lr=2.9e-04 trL=1.539 vL=2.303 trA=0.790 vA=0.655
[EfficientNetV2S] ep7/40 lr=2.9e-04 trL=1.464 vL=2.248 trA=0.806 vA=0.680
[EfficientNetV2S] ep8/40 lr=2.8e-04 trL=1.352 vL=2.222 trA=0.839 vA=0.697
[EfficientNetV2S] ep9/40 lr=2.8e-04 trL=1.341 vL=2.216 trA=0.846 vA=0.704
[EfficientNetV2S] ep10/40 lr=2.7e-04 trL=1.296 vL=2.246 trA=0.865 vA=0.681
[EfficientNetV2S] ep11/40 lr=2.6e-04 trL=1.246 vL=2.188 trA=0.886 vA=0.695
[EfficientNetV2S] ep12/40 lr=2.5e-04 trL=1.237 vL=2.280 trA=0.884 vA=0.684
[EfficientNetV2S] ep13/40 lr=2.4e-04 trL=1.182 vL=2.248 trA=0.911 vA=0.703
[EfficientNetV2S] ep14/40 lr=2.3e-

...

[MobileNetV3Large] ep1/40 lr=1.6e-04 trL=4.934 vL=4.875 trA=0.051 vA=0.174
[MobileNetV3Large] ep2/40 lr=3.0e-04 trL=3.511 vL=2.884 trA=0.325 vA=0.459
[MobileNetV3Large] ep3/40 lr=3.0e-04 trL=2.490 vL=2.708 trA=0.514 vA=0.540
[MobileNetV3Large] ep4/40 lr=3.0e-04 trL=2.011 vL=2.452 trA=0.625 vA=0.567
[MobileNetV3Large] ep5/40 lr=3.0e-04 trL=1.730 vL=2.314 trA=0.705 vA=0.625
[MobileNetV3Large] ep6/40 lr=2.9e-04 trL=1.550 vL=2.275 trA=0.748 vA=0.648
[MobileNetV3Large] ep7/40 lr=2.9e-04 trL=1.460 vL=2.214 trA=0.790 vA=0.661
[MobileNetV3Large] ep8/40 lr=2.8e-04 trL=1.363 vL=2.152 trA=0.816 vA=0.673
[MobileNetV3Large] ep9/40 lr=2.8e-04 trL=1.279 vL=2.223 trA=0.845 vA=0.657
[MobileNetV3Large] ep10/40 lr=2.7e-04 trL=1.247 vL=2.232 trA=0.863 vA=0.678
[MobileNetV3Large] ep11/40 lr=2.6e-04 trL=1.200 vL=2.182 trA=0.878 vA=0.700
[MobileNetV3Large] ep12/40 lr=2.5e-04 trL=1.183 vL=2.156 trA=0.885 vA=0.694
[MobileNetV3Large] ep13/40 lr=2.4e-04 trL=1.167 vL=2.278 trA=0.899 vA=0.696
[MobileNetV3Large] ep

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[ViTBase16] ep1/40 lr=2.0e-05 trL=4.791 vL=4.102 trA=0.066 vA=0.245
[ViTBase16] ep2/40 lr=3.5e-05 trL=2.764 vL=2.399 trA=0.465 vA=0.605
[ViTBase16] ep3/40 lr=5.0e-05 trL=1.845 vL=2.337 trA=0.673 vA=0.641
[ViTBase16] ep4/40 lr=5.0e-05 trL=1.684 vL=2.393 trA=0.731 vA=0.628
[ViTBase16] ep5/40 lr=5.0e-05 trL=1.426 vL=2.128 trA=0.804 vA=0.680
[ViTBase16] ep6/40 lr=4.9e-05 trL=1.320 vL=2.256 trA=0.841 vA=0.660
[ViTBase16] ep7/40 lr=4.9e-05 trL=1.210 vL=2.139 trA=0.881 vA=0.691
[ViTBase16] ep8/40 lr=4.8e-05 trL=1.162 vL=2.165 trA=0.893 vA=0.688
[ViTBase16] ep9/40 lr=4.7e-05 trL=1.134 vL=2.202 trA=0.903 vA=0.702
[ViTBase16] ep10/40 lr=4.6e-05 trL=1.096 vL=2.163 trA=0.914 vA=0.703
[ViTBase16] ep11/40 lr=4.4e-05 trL=1.070 vL=2.249 trA=0.922 vA=0.682
[ViTBase16] ep12/40 lr=4.3e-05 trL=1.050 vL=2.168 trA=0.932 vA=0.693
[ViTBase16] ep13/40 lr=4.2e-05 trL=1.026 vL=2.183 trA=0.939 vA=0.695
[ViTBase16] ep14/40 lr=4.0e-05 trL=1.029 vL=2.186 trA=0.934 vA=0.707
[ViTBase16] ep15/40 lr=3.8e-05 trL=0.986 vL

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

[SwinTiny] ep1/40 lr=2.0e-05 trL=5.000 vL=5.092 trA=0.021 vA=0.040
[SwinTiny] ep2/40 lr=3.5e-05 trL=4.122 vL=3.262 trA=0.199 vA=0.408
[SwinTiny] ep3/40 lr=5.0e-05 trL=2.763 vL=2.665 trA=0.473 vA=0.552
[SwinTiny] ep4/40 lr=5.0e-05 trL=2.214 vL=2.396 trA=0.596 vA=0.595
[SwinTiny] ep5/40 lr=5.0e-05 trL=1.817 vL=2.347 trA=0.694 vA=0.621
[SwinTiny] ep6/40 lr=4.9e-05 trL=1.618 vL=2.189 trA=0.750 vA=0.693
[SwinTiny] ep7/40 lr=4.9e-05 trL=1.470 vL=2.186 trA=0.788 vA=0.687
[SwinTiny] ep8/40 lr=4.8e-05 trL=1.389 vL=2.159 trA=0.825 vA=0.697
[SwinTiny] ep9/40 lr=4.7e-05 trL=1.302 vL=2.105 trA=0.855 vA=0.708
[SwinTiny] ep10/40 lr=4.6e-05 trL=1.262 vL=2.100 trA=0.868 vA=0.723
[SwinTiny] ep11/40 lr=4.4e-05 trL=1.212 vL=2.109 trA=0.886 vA=0.700
[SwinTiny] ep12/40 lr=4.3e-05 trL=1.163 vL=2.096 trA=0.898 vA=0.720
[SwinTiny] ep13/40 lr=4.2e-05 trL=1.134 vL=2.072 trA=0.913 vA=0.717
[SwinTiny] ep14/40 lr=4.0e-05 trL=1.119 vL=2.054 trA=0.917 vA=0.716
[SwinTiny] ep15/40 lr=3.8e-05 trL=1.091 vL=2.057 trA=0.92

## Classifiers-Only Lane B (seed 7) complete

All 8 classifiers trained and saved to `/kaggle/working/s1_results_seed7/`. Bring the
`result_*_seed7.pkl` files (and `*_best.pt` checkpoints, if you want them) back to
`revision_work/multiseed/` for aggregation with the already-collected seed-7 segmenter
results and the original SEED=42 checkpoints.